# Race gait analysis on Google Colab (efficient)

Pull the race straight from YouTube into Colab, pose-track your athlete on the
GPU, and turn her gait (ground contact, cadence, bounce, head sway) into a
running-economy and finish-time estimate.

**First:** *Runtime → Change runtime type → GPU (T4 is fine)*.

Downloading the YouTube clip here is far faster than uploading a file, and
vidssave.com just re-hosts the same YouTube video — so paste the YouTube URL
below and skip the download site entirely.

In [ ]:
# 1. Confirm the GPU and install everything (CV stack + YouTube downloader).
!nvidia-smi -L
!pip -q install ultralytics opencv-python-headless yt-dlp
%cd /content
![ -d dance-pose-tracker ] && (cd dance-pose-tracker && git pull) || git clone -b claude/athletic-performance-predictor-gmhkll https://github.com/henosss/dance-pose-tracker.git
%cd /content/dance-pose-tracker

## 2. Get the video
Paste the YouTube URL of the race. Optionally set `CLIP` to just the part you
care about (e.g. the last few laps) — downloading and processing only that is a
big speed-up. Or switch `SOURCE` to a direct `.mp4` link (vidssave) or a manual
upload.

In [ ]:
SOURCE      = 'youtube'                 # 'youtube' | 'url' | 'upload'\nYOUTUBE_URL = 'PASTE_THE_YOUTUBE_URL_HERE'\nDIRECT_URL  = ''                        # a direct .mp4 link (e.g. from vidssave.com)\nCLIP        = ''                        # '' = whole video, or e.g. '6:30-13:40' for the finish\n\nimport os\n!rm -f race.mp4 race.*.part\nif SOURCE == 'youtube':\n    assert not YOUTUBE_URL.startswith('PASTE'), 'Paste the real YouTube URL into YOUTUBE_URL first!'\n    # Robust: prefer a single progressive <=480p stream (no merge needed), then fall back.\n    FMT = 'b[height<=480][ext=mp4]/b[height<=480]/bv*[height<=480]+ba/best'\n    SECTIONS = f'--download-sections \"*{CLIP}\"' if CLIP else ''\n    !yt-dlp -f \"{FMT}\" {SECTIONS} --no-playlist --force-overwrites --merge-output-format mp4 -o race.mp4 \"{YOUTUBE_URL}\"\n    VIDEO = 'race.mp4'\nelif SOURCE == 'url':\n    !wget -O race.mp4 \"{DIRECT_URL}\"\n    VIDEO = 'race.mp4'\nelse:\n    from google.colab import files\n    VIDEO = next(iter(files.upload()))\n\n# Verify the download actually produced a playable video before going further.\nassert os.path.exists(VIDEO), f'{VIDEO} was not created — the download failed.'\nsize_kb = os.path.getsize(VIDEO) // 1024\nimport cv2\ncap = cv2.VideoCapture(VIDEO)\nframes = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); fps = cap.get(cv2.CAP_PROP_FPS); cap.release()\nprint(f'video: {VIDEO} — {size_kb} KB, {frames} frames @ {fps:.0f} fps, ~{frames/fps:.0f}s' if fps else f'{VIDEO}: {size_kb} KB')\nassert size_kb > 50 and frames > 0, 'Download looks empty/broken — check the yt-dlp output above and the URL.'

In [ ]:
# 3. Who/what are we analysing, and the speed knobs.
ATHLETE           = 'Senayet Getachew'   # must match a name in the dataset
ATHLETE_HEIGHT_CM = 165                  # calibrates pixels -> cm
EVENT             = '5000m'
GEAR              = 'super_spikes'
FATIGUE_ONSET     = 0.6                   # None = whole race; 0.6 = late-race fault

MODEL      = 'yolov8n-pose.pt'   # 'n' fastest; 's'/'m'/'x' slower + more accurate
IMGSZ      = 480                 # 480 matches a 480p source; 384 is faster
VID_STRIDE = 1                   # 2 = every other frame (~2x faster, coarser timing)
HALF       = True                # fp16 on GPU — big speed-up, set False on CPU

## 4. Quick trial, then the full run
The trial processes ~200 frames so you can read the fps before committing. If
it's slow: lower `IMGSZ` to 384 or set `VID_STRIDE = 2` above. The video is
decoded only once (shot cuts are found inline) and YOLO logging is silenced.

In [ ]:
from athlete_predictor.video import extract_segments
_ = extract_segments(VIDEO, model=MODEL, imgsz=IMGSZ, vid_stride=VID_STRIDE,
                     half=HALF, max_frames=200)
print('\nTrial done — if the fps looks OK, run the next cell for the whole clip.')

In [ ]:
shots = extract_segments(VIDEO, model=MODEL, imgsz=IMGSZ, vid_stride=VID_STRIDE, half=HALF)
print(f'{len(shots.shots)} camera shot(s) at {shots.fps:.0f} fps')

## 5. Find your athlete in each shot
Track IDs reset at every camera cut. Each preview has the runners' **track IDs
drawn on**. Find your athlete and note her ID per shot.

In [ ]:
import cv2, matplotlib.pyplot as plt
for i, img in enumerate(shots.previews):
    if img is None:
        continue
    plt.figure(figsize=(9, 5))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f'shot {i}  —  track IDs: {shots.track_ids(i)}')
    plt.axis('off'); plt.show()

In [ ]:
# EDIT THIS: map each shot index to your athlete's track ID. Skip shots where
# she isn't visible. Or set AUTO = True to take the longest track per shot.
AUTO = False
chosen = {
    0: 1,
    # 2: 3,
}
if AUTO:
    chosen = {i: shots.track_ids(i)[0] for i in range(len(shots.shots)) if shots.track_ids(i)}
    print('auto-picked:', chosen)

In [ ]:
# 6. Save her keypoints and run the gait -> economy -> time analysis.
from athlete_predictor.video import save_poses_json
from athlete_predictor.cli import main

save_poses_json('athlete_poses.json', shots, chosen,
                fps=shots.fps, athlete_height_cm=ATHLETE_HEIGHT_CM)
argv = ['analyze', '--poses', 'athlete_poses.json',
        '--athlete', ATHLETE, '--event', EVENT, '--gear', GEAR]
if FATIGUE_ONSET is not None:
    argv += ['--fatigue-onset', str(FATIGUE_ONSET)]
main(argv)

## 7. Bonus: equalized comparison and what-ifs

In [ ]:
from athlete_predictor.cli import main
main(['compare', '--event', '5000m', '--gear', 'super_spikes',
      '--athletes', 'Freweyni Hailu', 'Senayet Getachew'])
print()
main(['predict', '--athlete', ATHLETE, '--event', EVENT, '--gear', GEAR,
      '--fix', 'head_wobble', '--fatigue-onset', '0.6'])